# Section V Trade-off Evidence Lab (Full-Scan + JSON/Markdown Fusion)

This notebook builds the evidence layer for Section V (Performance Trade-off Analysis).
- Full scan: processed markdowns + O_ISAC JSON
- Variant-aware retrieval (lexical + fuzzy + LLM entailment)
- Two-model flow:
  - Pass-1 model (fast): broad classification over all hits
  - Pass-2 model (strict): only escalated uncertain hits
- Outputs: metric-governance tables, CRQ/Pareto candidates, modality trade-off slices

Usage note:
- Tune model names and per-model RPM in `# @title 3. Config`.
- Keep `RESUME=True` for long runs; checkpoints are under `analysis/V_ev_v1/checkpoints`.


In [1]:
# @title 1. Install Dependencies
!pip install -q groq rapidfuzz tqdm


In [2]:
# @title 2. Setup & Mount Drive
from google.colab import drive, userdata
import os, re, json, glob
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from rapidfuzz import fuzz

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST'
if os.path.exists(BASE_DIR):
    os.chdir(BASE_DIR)
    print('Working dir:', os.getcwd())
else:
    print('Path not found:', BASE_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working dir: /content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST


In [3]:
# @title 3. Config
PROCESSED_MD_DIR = Path('data/proc_markdowns')
JSON_DIR = Path('data/ext_res_v4')
UNIFIED_JSON = JSON_DIR / 'extraction_v4_unified.json'
OUTPUT_DIR = Path('analysis/V_ev_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_PROFILE = 'FULL_RESCAN'
NORMALIZE_LABELS = True

TARGET_PAPERS = None  # Full scan by default
LIMIT = None          # Full scan by default

# LLM execution controls (limit-aware)
LLM_CALLS = True

# Two-model strategy
MODEL_PASS1 = 'meta-llama/llama-4-scout-17b-16e-instruct'   # fast sweep
MODEL_PASS2 = 'llama-3.3-70b-versatile'                      # strict recheck
MODEL_VARIANT_GEN = MODEL_PASS1                              # variant generation model
USE_ESCALATION = True
ESCALATE_LABELS = {'INDIRECT', 'NONE', 'WEAK'}

# Per-model rate limits (requests/min)
RPM_BY_MODEL = {
    MODEL_PASS1: 120,
    MODEL_PASS2: 40,
    MODEL_VARIANT_GEN: 120,
}
DEFAULT_RPM = 30

MAX_RETRIES = 5
RETRY_BASE_SECONDS = 2.0

# Retrieval and batching controls
MAX_VARIANTS_PER_CONCEPT = 10
MAX_HITS_PER_CONCEPT_PER_PAPER = 5
MAX_CONTEXT_CHARS = 1200
CLASSIFY_CHUNK_SIZE = 4
BATCH_SIZE_PAPERS = 10

# Checkpoint controls
RESUME = True
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

SCHEMA_MAP_PATH = Path('analysis/II_sch_map.md')
GOV_PATH = Path('analysis/II_met_gov.md')

schema_text = SCHEMA_MAP_PATH.read_text(encoding='utf-8', errors='ignore')
gov_text = GOV_PATH.read_text(encoding='utf-8', errors='ignore')

print('Config ready. Output:', OUTPUT_DIR)
print('Run profile:', RUN_PROFILE)
print('Pass-1 model:', MODEL_PASS1)
print('Pass-2 model:', MODEL_PASS2)


Config ready. Output: analysis/V_ev_v2
Run profile: FULL_RESCAN
Pass-1 model: meta-llama/llama-4-scout-17b-16e-instruct
Pass-2 model: llama-3.3-70b-versatile


In [4]:
# @title 4. Load O_ISAC JSON Index
def load_json_index(json_dir: Path):
    index = {}
    for p in sorted(json_dir.glob('O_ISAC_*_v4.json')):
        paper_id = p.stem.replace('_v4','')
        try:
            index[paper_id] = json.loads(p.read_text(encoding='utf-8', errors='ignore'))
        except Exception as e:
            index[paper_id] = {'_error': str(e)}
    unified = None
    if UNIFIED_JSON.exists():
        unified = json.loads(UNIFIED_JSON.read_text(encoding='utf-8', errors='ignore'))
    return index, unified

json_index, unified_json = load_json_index(JSON_DIR)
print('JSON files loaded:', len(json_index))
print('Unified JSON:', 'yes' if unified_json else 'no')


JSON files loaded: 221
Unified JSON: yes


In [5]:
# @title 5. Load Processed Markdowns (Canonical per paper)
def canonical_md_path(paths, paper_id):
    scored = []
    for p in paths:
        p = Path(p)
        score = 0
        if (p.parent / 'visual_analysis.txt').exists():
            score += 3
        if p.parent.name == paper_id and p.parent.parent.name == paper_id:
            score += 2
        score += len(p.parts) * 0.1
        scored.append((score, p))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1] if scored else None

def load_processed_markdowns(target_ids=None, limit=None):
    search_path = PROCESSED_MD_DIR
    all_files = list(search_path.rglob('*.md'))
    md_files = [p for p in all_files if 'O_ISAC_' in p.name]

    grouped = {}
    for p in md_files:
        m = re.search(r'(O_ISAC_\d+)', p.name)
        if not m:
            continue
        paper_id = m.group(1)
        if target_ids and paper_id not in target_ids:
            continue
        grouped.setdefault(paper_id, []).append(p)

    records = []
    for i, (paper_id, paths) in enumerate(sorted(grouped.items())):
        if limit and i >= limit:
            break
        canon = canonical_md_path(paths, paper_id)
        if not canon:
            continue
        text = canon.read_text(encoding='utf-8', errors='ignore')
        lines = text.splitlines()
        va_path = canon.parent / 'visual_analysis.txt'
        va_text = va_path.read_text(encoding='utf-8', errors='ignore') if va_path.exists() else ''
        records.append({
            'paper_id': paper_id,
            'md_path': str(canon),
            'text': text,
            'lines': lines,
            'visual_analysis': va_text
        })
    return records

papers = load_processed_markdowns(target_ids=TARGET_PAPERS, limit=LIMIT)
print('Markdown papers loaded:', len(papers))


Markdown papers loaded: 221


In [6]:
# @title 6. Heading + Metric Helpers
import ast

def build_heading_map(lines):
    current = []
    heading_map = {}
    for i, line in enumerate(lines):
        if line.startswith('#'):
            level = len(line) - len(line.lstrip('#'))
            title = line.strip('#').strip()
            if level <= len(current):
                current = current[:level - 1]
            current.append(title)
        heading_map[i] = ' > '.join(current) if current else 'no_heading'
    return heading_map

def get_context(lines, idx, window=2):
    start = max(0, idx - window)
    end = min(len(lines), idx + window + 1)
    return '\n'.join(lines[start:end])

def to_float(x):
    try:
        if x is None:
            return None
        if isinstance(x, str) and not x.strip():
            return None
        return float(x)
    except Exception:
        return None

def contains_any(text, keywords):
    if not text:
        return False
    low = str(text).lower()
    return any(k in low for k in keywords)

MEDIUM_ALIAS_MAP = {
    'visible_light': 'wireless_vlc',
    'vlc': 'wireless_vlc',
    'rf': 'wireless_rf',
    'photo_thz': 'terahertz',
    'photonic_thz': 'terahertz',
}

def normalize_medium_label(medium_value):
    m = str(medium_value).lower().strip() if medium_value is not None else ''
    if not m:
        return 'unknown'
    return MEDIUM_ALIAS_MAP.get(m, m)

def normalize_medium(record):
    clsf = record.get('study_level', {}).get('classification', {}) if isinstance(record, dict) else {}
    medium = clsf.get('oisac_medium_class', 'unknown') if isinstance(clsf, dict) else 'unknown'
    return normalize_medium_label(medium)

def normalize_task_label(task_value):
    tokens = []
    if isinstance(task_value, list):
        tokens = [str(t).strip().lower() for t in task_value if str(t).strip()]
    elif isinstance(task_value, str):
        raw = task_value.strip()
        if raw.startswith('[') and raw.endswith(']'):
            try:
                parsed = ast.literal_eval(raw)
                if isinstance(parsed, list):
                    tokens = [str(t).strip().lower() for t in parsed if str(t).strip()]
                elif raw:
                    tokens = [raw.lower()]
            except Exception:
                tokens = [raw.lower()] if raw else []
        elif raw:
            tokens = [raw.lower()]
    elif task_value is not None:
        tokens = [str(task_value).strip().lower()]

    dedup = []
    seen = set()
    for t in tokens:
        if t and t not in seen:
            seen.add(t)
            dedup.append(t)

    if not dedup:
        return 'unknown'
    if len(dedup) == 1:
        return dedup[0]
    return '|_'.join(dedup)

def is_ranging_task(task_text):
    return contains_any(task_text, ['ranging', 'tof', 'fmcw', 'range'])

def is_fiber_task(task_text):
    return contains_any(task_text, ['das', 'otdr', 'ofdr', 'fiber', 'fibre'])

def has_electrical_rx_cue(rx_type):
    # Explicit receiver cues indicate post-detection electrical interpretation for SNR terms.
    return contains_any(rx_type, ['im/dd', 'direct', 'coherent', 'heterodyne', 'homodyne', 'envelope', 'photo', 'detector', 'pd', 'bpd'])

def extract_rate_bps(scn):
    cm = scn.get('comm_metrics', {}) if isinstance(scn, dict) else {}
    r_gbps = to_float(cm.get('data_rate_gbps')) if isinstance(cm, dict) else None
    if r_gbps is not None and r_gbps > 0:
        return r_gbps * 1e9, 'data_rate_gbps'
    return None, None

def scenario_metric_flags(record):
    medium = normalize_medium(record)
    raw_scenarios = record.get('scenario_level', []) if isinstance(record, dict) else []
    if isinstance(raw_scenarios, list):
        scenarios = [s for s in raw_scenarios if isinstance(s, dict)]
    elif isinstance(raw_scenarios, dict):
        scenarios = [raw_scenarios]
    else:
        scenarios = []

    rows = []
    for idx, scn in enumerate(scenarios):
        sm = scn.get('sensing_metrics', {}) if isinstance(scn, dict) else {}
        cm = scn.get('comm_metrics', {}) if isinstance(scn, dict) else {}
        rx = scn.get('receiver', {}) if isinstance(scn, dict) else {}
        tw = scn.get('tradeoff', {}) if isinstance(scn, dict) else {}

        task_raw = (sm.get('sensing_task_type') if isinstance(sm, dict) else '')
        task = normalize_task_label(task_raw)

        r_bps, r_src = extract_rate_bps(scn)
        drmin = to_float(sm.get('range_resolution_m')) if isinstance(sm, dict) else None
        dz = to_float(sm.get('spatial_resolution_m')) if isinstance(sm, dict) else None

        range_acc = to_float(sm.get('range_accuracy_m')) if isinstance(sm, dict) else None
        loc_err = to_float(sm.get('localization_error_m')) if isinstance(sm, dict) else None
        sigma_r = range_acc if range_acc is not None else loc_err

        crb = to_float(sm.get('crb_crlb_value')) if isinstance(sm, dict) else None
        crb_param = (sm.get('crb_parameter') if isinstance(sm, dict) else '') or ''

        osnr = to_float(cm.get('osnr_db')) if isinstance(cm, dict) else None
        snr = to_float(cm.get('snr_db')) if isinstance(cm, dict) else None
        rx_type = (rx.get('rx_detection_type') if isinstance(rx, dict) else '') or ''

        tradeoff_type = (tw.get('tradeoff_type') if isinstance(tw, dict) else '') or ''
        coupling_mode = (tw.get('coupling_mode') if isinstance(tw, dict) else '') or ''

        drmin_eligible = (
            drmin is not None and drmin > 0 and (
                medium in {
                    'wireless_fso', 'wireless_vlc', 'hybrid', 'terahertz', 'wireless',
                    'wireless_rf', 'wireless_retroreflective', 'retroreflective', 'retroreflective_optical'
                }
                or is_ranging_task(task)
            )
        )
        dz_eligible = (
            dz is not None and dz > 0 and (
                medium == 'cabled_fibre' or is_fiber_task(task)
            )
        )

        plane_mixed = (osnr is not None and snr is not None)
        imdd_osnr_conflict = contains_any(rx_type, ['im/dd', 'direct']) and (osnr is not None)
        snr_ambiguous = (snr is not None) and (osnr is None) and (not has_electrical_rx_cue(rx_type))

        dz_drmin_alias = False
        if drmin is not None and drmin > 0 and dz is not None and dz > 0 and (medium == 'cabled_fibre' or is_fiber_task(task)):
            den = max(abs(drmin), abs(dz), 1e-12)
            dz_drmin_alias = abs(drmin - dz) / den < 1e-9

        crq_candidate = r_bps is not None and drmin_eligible
        governance_blocked = plane_mixed or imdd_osnr_conflict or snr_ambiguous or dz_drmin_alias
        crq_eligible = crq_candidate and (not governance_blocked)
        crq_delta = (r_bps / drmin) if crq_eligible and drmin and drmin > 0 else None

        rows.append({
            'scenario_index': idx,
            'medium': medium,
            'task_type': task,
            'r_bps': r_bps,
            'r_source': r_src,
            'drmin_m': drmin,
            'drmin_eligible': drmin_eligible,
            'dz_m': dz,
            'dz_eligible': dz_eligible,
            'sigma_r_m': sigma_r,
            'crb_value': crb,
            'crb_param': str(crb_param),
            'osnr_db': osnr,
            'snr_db': snr,
            'rx_detection_type': str(rx_type),
            'tradeoff_type': str(tradeoff_type).strip().lower(),
            'coupling_mode': str(coupling_mode).strip().lower(),
            'plane_mixed': plane_mixed,
            'imdd_osnr_conflict': imdd_osnr_conflict,
            'snr_ambiguous': snr_ambiguous,
            'dz_drmin_alias': dz_drmin_alias,
            'governance_blocked': governance_blocked,
            'crq_candidate': crq_candidate,
            'crq_eligible': crq_eligible,
            'crq_delta_bps_per_m': crq_delta,
        })

    return rows


In [7]:
# @title 7. Groq Client + Variant Generator (Cache + Per-Model Rate Limit)
from groq import Groq
from collections import deque
import time
import random

VARIANT_CACHE = OUTPUT_DIR / 'variant_cache.json'
if VARIANT_CACHE.exists():
    variant_cache = json.loads(VARIANT_CACHE.read_text(encoding='utf-8'))
else:
    variant_cache = {}

_GROQ_CLIENT = None
REQUEST_LOG_BY_MODEL = {}


def get_groq_client():
    global _GROQ_CLIENT
    if _GROQ_CLIENT is not None:
        return _GROQ_CLIENT

    try:
        api_key = userdata.get('GROQ_API_KEY')
    except Exception:
        api_key = os.environ.get('GROQ_API_KEY')

    if not api_key:
        raise ValueError('GROQ_API_KEY not found in Colab Secrets or env.')

    _GROQ_CLIENT = Groq(api_key=api_key)
    return _GROQ_CLIENT


def get_model_rpm(model_name):
    return RPM_BY_MODEL.get(model_name, DEFAULT_RPM)


def throttle_requests(model_name):
    rpm = get_model_rpm(model_name)
    if rpm <= 0:
        return

    q = REQUEST_LOG_BY_MODEL.setdefault(model_name, deque())
    now = time.time()

    while q and now - q[0] > 60:
        q.popleft()

    if len(q) >= rpm:
        wait_s = 60 - (now - q[0]) + 0.1
        wait_s = max(wait_s, 0.1)
        print(f'Rate limit guard ({model_name}): sleeping {wait_s:.1f}s')
        time.sleep(wait_s)
        now = time.time()
        while q and now - q[0] > 60:
            q.popleft()

    q.append(time.time())


def safe_chat_completion(model_name, messages, expect_json=False, temperature=0.2):
    if not LLM_CALLS:
        return None

    client = get_groq_client()

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            throttle_requests(model_name)
            kwargs = {
                'model': model_name,
                'messages': messages,
                'temperature': temperature
            }
            if expect_json:
                kwargs['response_format'] = {'type': 'json_object'}

            resp = client.chat.completions.create(**kwargs)
            return resp.choices[0].message.content
        except Exception as e:
            if attempt >= MAX_RETRIES:
                print(f'LLM call failed ({model_name}) after {MAX_RETRIES} attempts: {e}')
                return None
            sleep_s = RETRY_BASE_SECONDS * (2 ** (attempt - 1)) + random.uniform(0.0, 0.5)
            print(f'LLM retry ({model_name}) {attempt}/{MAX_RETRIES}: {e}; sleeping {sleep_s:.1f}s')
            time.sleep(sleep_s)


def get_variants(concept):
    if concept in variant_cache:
        vals = variant_cache[concept]
        return vals[:MAX_VARIANTS_PER_CONCEPT]

    if not LLM_CALLS:
        vals = [concept]
        variant_cache[concept] = vals
        return vals

    system_prompt = (
        'You generate lexical variants and paraphrases for evidence retrieval. '
        'Return compact JSON: {"variants": ["..."]}.'
    )
    user_prompt = (
        f'Concept: {concept}\n'
        'Return up to 12 variants including synonyms, abbreviations, paraphrases, and morphological forms. '
        'Keep each variant short.'
    )

    content = safe_chat_completion(
        model_name=MODEL_VARIANT_GEN,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ],
        expect_json=True,
        temperature=0.2
    )

    variants = [concept]
    if content:
        try:
            data = json.loads(content)
            llm_vars = data.get('variants', [])
            if isinstance(llm_vars, list):
                for item in llm_vars:
                    if isinstance(item, str) and item.strip():
                        variants.append(item.strip())
        except Exception:
            pass

    dedup = []
    seen = set()
    for v in variants:
        key = v.lower().strip()
        if not key or key in seen:
            continue
        seen.add(key)
        dedup.append(v)

    dedup = dedup[:MAX_VARIANTS_PER_CONCEPT]
    variant_cache[concept] = dedup
    VARIANT_CACHE.write_text(json.dumps(variant_cache, ensure_ascii=False, indent=2), encoding='utf-8')
    return dedup


In [8]:
# @title 8. Retrieval + Entailment Classification (Two-Model, Batched, Checkpointed)
def scan_lines_for_variants(lines, variants, fuzzy_threshold=85):
    hits = []
    for i, line in enumerate(lines):
        text = line.strip()
        if not text:
            continue
        low = text.lower()
        for v in variants:
            vlow = v.lower()
            if vlow in low:
                hits.append((i, line, v, 'lexical'))
                break
            score = fuzz.partial_ratio(vlow, low)
            if score >= fuzzy_threshold:
                hits.append((i, line, v, f'fuzzy:{score}'))
                break
    return hits


def clip_text(text, max_chars=MAX_CONTEXT_CHARS):
    if text is None:
        return ''
    text = str(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + ' ...'


def chunk_list(items, n):
    for i in range(0, len(items), n):
        yield items[i:i+n]


def parse_batch_results(content, n):
    fallback = [{'label': 'WEAK', 'rationale': 'LLM parse failed'} for _ in range(n)]
    if not content:
        return fallback

    try:
        parsed = json.loads(content)
        results = parsed.get('results', [])
        mapped = {int(r['idx']): r for r in results if isinstance(r, dict) and 'idx' in r}
        out = []
        for i in range(n):
            r = mapped.get(i)
            if not r:
                out.append({'label': 'WEAK', 'rationale': 'No label'})
                continue
            label = str(r.get('label', 'WEAK')).upper().strip()
            if label not in {'DIRECT', 'INDIRECT', 'NONE'}:
                label = 'WEAK'
            out.append({'label': label, 'rationale': str(r.get('rationale', ''))})
        return out
    except Exception:
        return fallback


def classify_with_model(concept, contexts, model_name, hint_labels=None):
    compact = []
    for i, ctx in enumerate(contexts):
        row = {'idx': i, 'context': clip_text(ctx)}
        if hint_labels and i < len(hint_labels):
            row['hint_label'] = hint_labels[i]
        compact.append(row)

    system_prompt = (
        'You are an evidence auditor. '
        'For each snippet, decide if the concept is DIRECT, INDIRECT, or NONE. '
        'Return strict JSON object with key "results": '
        '[{"idx":0,"label":"DIRECT|INDIRECT|NONE","rationale":"..."}]'
    )
    user_prompt = (
        f'Concept: {concept}\n'
        f'Snippets JSON:\n{json.dumps(compact, ensure_ascii=False)}'
    )

    content = safe_chat_completion(
        model_name=model_name,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ],
        expect_json=True,
        temperature=0.1
    )

    return parse_batch_results(content, len(contexts))


def classify_hits_batch(concept, contexts):
    if not contexts:
        return []

    if not LLM_CALLS:
        return [{
            'label': 'WEAK',
            'rationale': 'LLM disabled',
            'label_pass1': 'WEAK',
            'rationale_pass1': 'LLM disabled',
            'model_pass1': MODEL_PASS1,
            'label_pass2': '',
            'rationale_pass2': '',
            'model_pass2': '',
            'escalated': False
        } for _ in contexts]

    pass1 = classify_with_model(concept, contexts, MODEL_PASS1)
    out = []
    for r in pass1:
        out.append({
            'label': r.get('label', 'WEAK'),
            'rationale': r.get('rationale', ''),
            'label_pass1': r.get('label', 'WEAK'),
            'rationale_pass1': r.get('rationale', ''),
            'model_pass1': MODEL_PASS1,
            'label_pass2': '',
            'rationale_pass2': '',
            'model_pass2': '',
            'escalated': False
        })

    if USE_ESCALATION:
        idxs = [i for i, r in enumerate(out) if r['label'] in ESCALATE_LABELS]
        if idxs:
            sub_contexts = [contexts[i] for i in idxs]
            hints = [out[i]['label_pass1'] for i in idxs]
            pass2 = classify_with_model(concept, sub_contexts, MODEL_PASS2, hint_labels=hints)

            for j, i in enumerate(idxs):
                r2 = pass2[j]
                out[i]['label_pass2'] = r2.get('label', 'WEAK')
                out[i]['rationale_pass2'] = r2.get('rationale', '')
                out[i]['model_pass2'] = MODEL_PASS2
                out[i]['escalated'] = True

                if r2.get('label') in {'DIRECT', 'INDIRECT', 'NONE'}:
                    out[i]['label'] = r2.get('label')
                    out[i]['rationale'] = r2.get('rationale', '')

    return out


def classify_hits_chunked(concept, contexts):
    out = []
    for chunk in chunk_list(contexts, CLASSIFY_CHUNK_SIZE):
        out.extend(classify_hits_batch(concept, chunk))
    return out


def append_rows_csv(out_csv, rows):
    if not rows:
        return
    df_new = pd.DataFrame(rows)
    if out_csv.exists():
        df_new.to_csv(out_csv, mode='a', header=False, index=False)
    else:
        df_new.to_csv(out_csv, index=False)


def checkpoint_path(section_name):
    return CHECKPOINT_DIR / f'{section_name}_done_ids.json'


def load_done_ids(section_name):
    if not RESUME:
        return set()
    cp = checkpoint_path(section_name)
    if not cp.exists():
        return set()
    try:
        data = json.loads(cp.read_text(encoding='utf-8'))
        return set(data)
    except Exception:
        return set()


def save_done_ids(section_name, done_ids):
    cp = checkpoint_path(section_name)
    cp.write_text(json.dumps(sorted(list(done_ids)), ensure_ascii=False, indent=2), encoding='utf-8')


def process_in_batches(records):
    for batch in chunk_list(records, BATCH_SIZE_PAPERS):
        yield batch


def llm_fields_from_cls(cls):
    return {
        'llm_model_pass1': cls.get('model_pass1', ''),
        'llm_label_pass1': cls.get('label_pass1', ''),
        'llm_model_pass2': cls.get('model_pass2', ''),
        'llm_label_pass2': cls.get('label_pass2', ''),
        'llm_escalated': cls.get('escalated', False),
    }


In [9]:
# @title 9. Section 5A Evidence (Trade-off Concepts + Eligibility)
section_name = 'section5A'
out_csv = OUTPUT_DIR / f'{section_name}_evidence.csv'

done_ids = load_done_ids(section_name)
pending = [p for p in papers if p['paper_id'] not in done_ids]
print(f'{section_name}: pending papers = {len(pending)}')

concepts = [
    'trade-off',
    'multiobjective optimization',
    'pareto frontier',
    'weighted sum optimization',
    'power allocation',
    'communication-centric objective',
    'sensing-centric objective',
    'joint design operating point',
    'capacity-resolution quotient',
    'resource coupling',
]
concept_variants = {c: get_variants(c) for c in concepts}

for batch in process_in_batches(pending):
    batch_rows = []
    for paper in tqdm(batch, desc=f'{section_name} batch'):
        paper_id = paper['paper_id']
        lines = paper['lines']
        heading_map = build_heading_map(lines)
        record = json_index.get(paper_id, {})

        for concept, variants in concept_variants.items():
            hits = scan_lines_for_variants(lines, variants)
            hits = hits[:MAX_HITS_PER_CONCEPT_PER_PAPER]
            contexts = [get_context(lines, idx) for idx, _, _, _ in hits]
            cls_all = classify_hits_chunked(concept, contexts)

            for (hit, cls) in zip(hits, cls_all):
                idx, line, variant, match_type = hit
                batch_rows.append({
                    'paper_id': paper_id,
                    'section': '5A',
                    'concept': concept,
                    'variant': variant,
                    'match_type': match_type,
                    'strength': cls.get('label', 'WEAK'),
                    'rationale': cls.get('rationale', ''),
                    'quote': line.strip(),
                    'line_start': idx + 1,
                    'line_end': idx + 1,
                    'heading_path': heading_map.get(idx, 'no_heading'),
                    'json_path': '',
                    'json_value': '',
                    **llm_fields_from_cls(cls),
                })

        if isinstance(record, dict):
            flags = scenario_metric_flags(record)
            summary = {
                'n_scenarios': len(flags),
                'n_rate': sum(1 for f in flags if f.get('r_bps') is not None),
                'n_drmin_eligible': sum(1 for f in flags if f.get('drmin_eligible')),
                'n_dz_eligible': sum(1 for f in flags if f.get('dz_eligible')),
                'n_sigma_r': sum(1 for f in flags if f.get('sigma_r_m') is not None),
                'n_crb': sum(1 for f in flags if f.get('crb_value') is not None),
                'n_crq_candidate': sum(1 for f in flags if f.get('crq_candidate')),
                'n_crq_eligible': sum(1 for f in flags if f.get('crq_eligible')),
                'n_plane_mixed': sum(1 for f in flags if f.get('plane_mixed')),
                'n_imdd_osnr_conflict': sum(1 for f in flags if f.get('imdd_osnr_conflict')),
                'n_snr_ambiguous': sum(1 for f in flags if f.get('snr_ambiguous')),
                'n_dz_drmin_alias': sum(1 for f in flags if f.get('dz_drmin_alias')),
                'n_governance_blocked': sum(1 for f in flags if f.get('governance_blocked')),
            }
            batch_rows.append({
                'paper_id': paper_id,
                'section': '5A',
                'concept': 'json:tradeoff_eligibility',
                'variant': '',
                'match_type': 'json',
                'strength': 'DIRECT',
                'rationale': 'Structured metric-eligibility summary',
                'quote': '',
                'line_start': '',
                'line_end': '',
                'heading_path': '',
                'json_path': 'scenario_level[*].comm_metrics/sensing_metrics/tradeoff',
                'json_value': json.dumps(summary, ensure_ascii=False),
                'llm_model_pass1': 'json',
                'llm_label_pass1': 'DIRECT',
                'llm_model_pass2': '',
                'llm_label_pass2': '',
                'llm_escalated': False,
            })

        done_ids.add(paper_id)

    append_rows_csv(out_csv, batch_rows)
    save_done_ids(section_name, done_ids)
    print(f'{section_name}: wrote {len(batch_rows)} rows; done={len(done_ids)}')

print('Saved:', out_csv)


section5A: pending papers = 0
Saved: analysis/V_ev_v2/section5A_evidence.csv


In [10]:
# @title 10. Section 5B Evidence (Metric Governance Audit Table)
rows = []
for paper in tqdm(papers, desc='section5B'):
    paper_id = paper['paper_id']
    record = json_index.get(paper_id, {})
    if not isinstance(record, dict):
        continue

    flags = scenario_metric_flags(record)
    rows.append({
        'paper_id': paper_id,
        'n_scenarios': len(flags),
        'n_rate': sum(1 for f in flags if f.get('r_bps') is not None),
        'n_drmin': sum(1 for f in flags if f.get('drmin_m') is not None),
        'n_drmin_eligible': sum(1 for f in flags if f.get('drmin_eligible')),
        'n_dz': sum(1 for f in flags if f.get('dz_m') is not None),
        'n_dz_eligible': sum(1 for f in flags if f.get('dz_eligible')),
        'n_sigma_r': sum(1 for f in flags if f.get('sigma_r_m') is not None),
        'n_crb': sum(1 for f in flags if f.get('crb_value') is not None),
        'n_osnr': sum(1 for f in flags if f.get('osnr_db') is not None),
        'n_snr': sum(1 for f in flags if f.get('snr_db') is not None),
        'n_plane_mixed': sum(1 for f in flags if f.get('plane_mixed')),
        'n_imdd_osnr_conflict': sum(1 for f in flags if f.get('imdd_osnr_conflict')),
        'n_snr_ambiguous': sum(1 for f in flags if f.get('snr_ambiguous')),
        'n_dz_drmin_alias': sum(1 for f in flags if f.get('dz_drmin_alias')),
        'n_governance_blocked': sum(1 for f in flags if f.get('governance_blocked')),
        'n_crq_candidate': sum(1 for f in flags if f.get('crq_candidate')),
        'n_crq_eligible': sum(1 for f in flags if f.get('crq_eligible')),
    })

df = pd.DataFrame(rows)
out_csv = OUTPUT_DIR / 's5b_met_gov.csv'
df.to_csv(out_csv, index=False)
print('Saved:', out_csv, 'rows=', len(df))


section5B: 100%|██████████| 221/221 [00:00<00:00, 15844.34it/s]


Saved: analysis/V_ev_v2/s5b_met_gov.csv rows= 221


In [11]:
# @title 11. Section 5C Evidence (CRQ + Pareto Candidates)
point_rows = []
mention_rows = []

mention_concepts = [
    'rate-resolution trade-off',
    'pareto optimal',
    'capacity-resolution quotient',
    'fisher information constraint',
    'resource allocation trade-off',
]
mention_variants = {c: get_variants(c) for c in mention_concepts}

for paper in tqdm(papers, desc='section5C'):
    paper_id = paper['paper_id']
    record = json_index.get(paper_id, {})
    lines = paper['lines']
    heading_map = build_heading_map(lines)

    if isinstance(record, dict):
        flags = scenario_metric_flags(record)
        for f in flags:
            point_rows.append({
                'paper_id': paper_id,
                'scenario_index': f.get('scenario_index'),
                'medium': f.get('medium'),
                'task_type': f.get('task_type'),
                'tradeoff_type': f.get('tradeoff_type'),
                'coupling_mode': f.get('coupling_mode'),
                'r_bps': f.get('r_bps'),
                'drmin_m': f.get('drmin_m'),
                'drmin_eligible': f.get('drmin_eligible'),
                'dz_m': f.get('dz_m'),
                'sigma_r_m': f.get('sigma_r_m'),
                'crb_value': f.get('crb_value'),
                'osnr_db': f.get('osnr_db'),
                'snr_db': f.get('snr_db'),
                'rx_detection_type': f.get('rx_detection_type'),
                'crq_candidate': f.get('crq_candidate'),
                'crq_eligible': f.get('crq_eligible'),
                'crq_delta_bps_per_m': f.get('crq_delta_bps_per_m'),
                'plane_mixed': f.get('plane_mixed'),
                'imdd_osnr_conflict': f.get('imdd_osnr_conflict'),
                'snr_ambiguous': f.get('snr_ambiguous'),
                'dz_drmin_alias': f.get('dz_drmin_alias'),
                'governance_blocked': f.get('governance_blocked'),
            })

    for concept, variants in mention_variants.items():
        hits = scan_lines_for_variants(lines, variants)
        hits = hits[:MAX_HITS_PER_CONCEPT_PER_PAPER]
        contexts = [get_context(lines, idx) for idx, _, _, _ in hits]
        cls_all = classify_hits_chunked(concept, contexts)

        for (hit, cls) in zip(hits, cls_all):
            idx, line, variant, match_type = hit
            mention_rows.append({
                'paper_id': paper_id,
                'section': '5C',
                'concept': concept,
                'variant': variant,
                'match_type': match_type,
                'strength': cls.get('label', 'WEAK'),
                'rationale': cls.get('rationale', ''),
                'quote': line.strip(),
                'line_start': idx + 1,
                'line_end': idx + 1,
                'heading_path': heading_map.get(idx, 'no_heading'),
                **llm_fields_from_cls(cls),
            })

points_csv = OUTPUT_DIR / 'section5C_tradeoff_points.csv'
mentions_csv = OUTPUT_DIR / 's5c_trade_mnts.csv'

pd.DataFrame(point_rows).to_csv(points_csv, index=False)
pd.DataFrame(mention_rows).to_csv(mentions_csv, index=False)

print('Saved:', points_csv, 'rows=', len(point_rows))
print('Saved:', mentions_csv, 'rows=', len(mention_rows))


section5C:  28%|██▊       | 62/221 [00:42<03:42,  1.40s/it]

Rate limit guard (llama-3.3-70b-versatile): sleeping 19.6s


section5C:  30%|██▉       | 66/221 [01:02<08:23,  3.25s/it]

Rate limit guard (llama-3.3-70b-versatile): sleeping 0.7s


section5C:  45%|████▍     | 99/221 [01:15<01:46,  1.15it/s]

...

Rate limit guard (llama-3.3-70b-versatile): sleeping 0.6s
Rate limit guard (llama-3.3-70b-versatile): sleeping 0.2s
Rate limit guard (llama-3.3-70b-versatile): sleeping 0.4s


section5C:  72%|███████▏  | 160/221 [02:06<02:53,  2.84s/it]

Rate limit guard (llama-3.3-70b-versatile): sleeping 0.1s


section5C: 100%|██████████| 221/221 [02:28<00:00,  1.49it/s]


Saved: analysis/V_ev_v2/section5C_tradeoff_points.csv rows= 226
Saved: analysis/V_ev_v2/s5c_trade_mnts.csv rows= 188


In [12]:
# @title 12. Section 5D Evidence (Modality-Sliced Trade-off Tables)
points_csv = OUTPUT_DIR / 'section5C_tradeoff_points.csv'
if not points_csv.exists():
    raise FileNotFoundError(f'Missing file: {points_csv}')

def to_bool(v):
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return bool(v)
    s = str(v).strip().lower()
    return s in {'1', 'true', 'yes', 'y'}

df = pd.read_csv(points_csv)
mask_crq = df['crq_eligible'].map(to_bool) if 'crq_eligible' in df.columns else pd.Series([False] * len(df))
df_valid = df[mask_crq].copy()
if df_valid.empty:
    mask_drmin = df['drmin_eligible'].map(to_bool) if 'drmin_eligible' in df.columns else pd.Series([False] * len(df))
    df_valid = df[mask_drmin].copy()

for col in ['r_bps', 'drmin_m', 'sigma_r_m', 'crq_delta_bps_per_m']:
    if col in df_valid.columns:
        df_valid[col] = pd.to_numeric(df_valid[col], errors='coerce')

if 'medium' in df_valid.columns:
    df_valid['medium'] = df_valid['medium'].map(normalize_medium_label).fillna('unknown')

grouped = df_valid.groupby('medium', dropna=False).agg(
    n_points=('paper_id', 'count'),
    n_unique_papers=('paper_id', 'nunique'),
    median_r_bps=('r_bps', 'median'),
    median_drmin_m=('drmin_m', 'median'),
    median_sigma_r_m=('sigma_r_m', 'median'),
    median_crq=('crq_delta_bps_per_m', 'median'),
    p90_crq=('crq_delta_bps_per_m', lambda s: s.quantile(0.9) if len(s.dropna()) else float('nan')),
).reset_index()

grouped = grouped.sort_values('n_points', ascending=False).reset_index(drop=True)
out_csv = OUTPUT_DIR / 'section5D_modality_slices.csv'
grouped.to_csv(out_csv, index=False)
print('Saved:', out_csv, 'rows=', len(grouped))


Saved: analysis/V_ev_v2/section5D_modality_slices.csv rows= 4


In [13]:
# @title 13. Section 5E Summary Outputs (Pareto + Final Summary)
points_csv = OUTPUT_DIR / 'section5C_tradeoff_points.csv'
slices_csv = OUTPUT_DIR / 'section5D_modality_slices.csv'

if not points_csv.exists():
    raise FileNotFoundError(f'Missing file: {points_csv}')

def to_bool(v):
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return bool(v)
    s = str(v).strip().lower()
    return s in {'1', 'true', 'yes', 'y'}

df = pd.read_csv(points_csv)
for col in ['r_bps', 'drmin_m', 'crq_delta_bps_per_m']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

mask = df['crq_eligible'].map(to_bool) if 'crq_eligible' in df.columns else pd.Series([False] * len(df))
if 'governance_blocked' in df.columns:
    mask = mask & (~df['governance_blocked'].map(to_bool))

valid = df[mask].copy()
valid = valid.dropna(subset=['r_bps', 'drmin_m'])

pareto_idx = []
records = valid[['r_bps', 'drmin_m']].to_numpy() if len(valid) else []
for i in range(len(records)):
    r_i, d_i = records[i]
    dominated = False
    for j in range(len(records)):
        if i == j:
            continue
        r_j, d_j = records[j]
        if (r_j >= r_i and d_j <= d_i) and (r_j > r_i or d_j < d_i):
            dominated = True
            break
    if not dominated:
        pareto_idx.append(i)

pareto = valid.iloc[pareto_idx].copy() if len(valid) else valid.copy()
pareto = pareto.sort_values(['r_bps', 'drmin_m'], ascending=[False, True])

pareto_csv = OUTPUT_DIR / 'section5E_pareto_points.csv'
pareto.to_csv(pareto_csv, index=False)

summary_rows = [{
    'n_total_points': int(len(df)),
    'n_valid_crq_points': int(len(valid)),
    'n_pareto_points': int(len(pareto)),
    'max_crq': float(valid['crq_delta_bps_per_m'].max()) if len(valid) and valid['crq_delta_bps_per_m'].notna().any() else float('nan'),
    'median_crq': float(valid['crq_delta_bps_per_m'].median()) if len(valid) and valid['crq_delta_bps_per_m'].notna().any() else float('nan'),
}]

summary_csv = OUTPUT_DIR / 'section5E_summary_table.csv'
pd.DataFrame(summary_rows).to_csv(summary_csv, index=False)

summary_json = OUTPUT_DIR / 'section5E_summary.json'
summary_json.write_text(json.dumps(summary_rows[0], indent=2), encoding='utf-8')

if slices_csv.exists():
    print('Slices available:', slices_csv)
print('Saved:', pareto_csv)
print('Saved:', summary_csv)
print('Saved:', summary_json)


Slices available: analysis/V_ev_v2/section5D_modality_slices.csv
Saved: analysis/V_ev_v2/section5E_pareto_points.csv
Saved: analysis/V_ev_v2/section5E_summary_table.csv
Saved: analysis/V_ev_v2/section5E_summary.json


In [14]:
# @title 14. Post-Processing Artifacts (Section 5)
import hashlib
from collections import defaultdict

def as_int_or_blank(x):
    try:
        if pd.isna(x) or x == '':
            return ''
        return int(float(x))
    except Exception:
        return ''

def to_bool(v):
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return bool(v)
    return str(v).strip().lower() in {'1', 'true', 'yes', 'y'}

p5a = OUTPUT_DIR / 'section5A_evidence.csv'
p5m = OUTPUT_DIR / 's5c_trade_mnts.csv'
p5p = OUTPUT_DIR / 'section5C_tradeoff_points.csv'
p5b = OUTPUT_DIR / 's5b_met_gov.csv'

df5a = pd.read_csv(p5a) if p5a.exists() else pd.DataFrame()
df5m = pd.read_csv(p5m) if p5m.exists() else pd.DataFrame()
df5p = pd.read_csv(p5p) if p5p.exists() else pd.DataFrame()
df5b = pd.read_csv(p5b) if p5b.exists() else pd.DataFrame()

# retrieval_hits.jsonl
out = OUTPUT_DIR / 'retrieval_hits.jsonl'
with out.open('w', encoding='utf-8') as f:
    for sec,df in [('5A',df5a),('5C',df5m)]:
        if df.empty:
            continue
        for _,r in df.iterrows():
            mt = str(r.get('match_type',''))
            if mt.lower() == 'json':
                continue
            rec = {
                'paper_id': str(r.get('paper_id','')),
                'section': sec,
                'concept': str(r.get('concept','')),
                'variant': str(r.get('variant','')),
                'match_type': mt,
                'quote': str(r.get('quote','')),
                'line_start': as_int_or_blank(r.get('line_start','')),
                'line_end': as_int_or_blank(r.get('line_end','')),
                'heading_path': str(r.get('heading_path','')),
                'strength': str(r.get('strength','')),
                'rationale': str(r.get('rationale','')),
            }
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print('Saved:', out)

# anchor_table.csv
rows = []
for sec,df in [('5A',df5a),('5C',df5m)]:
    if df.empty:
        continue
    for _,r in df.iterrows():
        concept = str(r.get('concept','')).strip()
        claim_key = f'{sec}|{concept}'
        claim_id = hashlib.sha1(claim_key.encode('utf-8')).hexdigest()[:12]
        rows.append({
            'claim_id': claim_id,
            'claim_key': claim_key,
            'section': sec,
            'paper_id': str(r.get('paper_id','')),
            'concept': concept,
            'variant': str(r.get('variant','')),
            'match_type': str(r.get('match_type','')),
            'strength': str(r.get('strength','')),
            'rationale': str(r.get('rationale','')),
            'quote': str(r.get('quote','')),
            'line_start': as_int_or_blank(r.get('line_start','')),
            'line_end': as_int_or_blank(r.get('line_end','')),
            'heading_path': str(r.get('heading_path','')),
            'json_path': str(r.get('json_path','')),
            'json_value': str(r.get('json_value','')),
        })
if rows:
    anc = pd.DataFrame(rows)
    agg = anc.assign(is_direct=anc['strength'].eq('DIRECT'), is_indirect=anc['strength'].eq('INDIRECT')).groupby('claim_id',as_index=False)[['is_direct','is_indirect']].sum()
    agg['claim_supported'] = (agg['is_direct']>=1) | (agg['is_indirect']>=2)
    anc = anc.merge(agg[['claim_id','claim_supported']], on='claim_id', how='left')
else:
    anc = pd.DataFrame(columns=['claim_id','claim_key','section','paper_id','concept','variant','match_type','strength','rationale','quote','line_start','line_end','heading_path','json_path','json_value','claim_supported'])
anc_path = OUTPUT_DIR / 'anchor_table.csv'
anc.to_csv(anc_path, index=False)
print('Saved:', anc_path)

# evidence_graph.jsonl + cluster_map.csv
anchors_by_paper = defaultdict(list)
for _,r in anc.iterrows():
    anchors_by_paper[str(r.get('paper_id',''))].append({'section':str(r.get('section','')),'concept':str(r.get('concept','')),'strength':str(r.get('strength',''))})

gov_by_paper = {}
if not df5b.empty and 'paper_id' in df5b.columns:
    for _,r in df5b.iterrows():
        pid = str(r.get('paper_id',''))
        gov_by_paper[pid] = {
            'n_scenarios': int(r.get('n_scenarios',0) or 0),
            'n_crq_candidate': int(r.get('n_crq_candidate',0) or 0),
            'n_crq_eligible': int(r.get('n_crq_eligible',0) or 0),
            'n_plane_mixed': int(r.get('n_plane_mixed',0) or 0),
            'n_imdd_osnr_conflict': int(r.get('n_imdd_osnr_conflict',0) or 0),
            'n_snr_ambiguous': int(r.get('n_snr_ambiguous',0) or 0),
            'n_dz_drmin_alias': int(r.get('n_dz_drmin_alias',0) or 0),
            'n_governance_blocked': int(r.get('n_governance_blocked',0) or 0),
        }

points_by_paper = defaultdict(lambda: {'n_points':0,'n_crq_eligible_points':0})
if not df5p.empty and 'paper_id' in df5p.columns:
    for _,r in df5p.iterrows():
        pid = str(r.get('paper_id',''))
        points_by_paper[pid]['n_points'] += 1
        if to_bool(r.get('crq_eligible',False)):
            points_by_paper[pid]['n_crq_eligible_points'] += 1

graph = OUTPUT_DIR / 'evidence_graph.jsonl'
with graph.open('w', encoding='utf-8') as f:
    cluster_rows = []
    for pid,j in sorted(json_index.items()):
        flags = scenario_metric_flags(j) if isinstance(j,dict) else []
        medium = flags[0]['medium'] if flags else 'unknown'
        task = flags[0]['task_type'] if flags else 'unknown'
        rec = {
            'paper_id': pid,
            'structured': {
                'medium': medium,
                'task_type': task,
                'tradeoff_types': sorted({str(x.get('tradeoff_type','')) for x in flags if str(x.get('tradeoff_type','')).strip()}),
                'coupling_modes': sorted({str(x.get('coupling_mode','')) for x in flags if str(x.get('coupling_mode','')).strip()}),
            },
            'governance': gov_by_paper.get(pid, {
                'n_scenarios': len(flags),
                'n_crq_candidate': sum(1 for x in flags if x.get('crq_candidate')),
                'n_crq_eligible': sum(1 for x in flags if x.get('crq_eligible')),
                'n_plane_mixed': sum(1 for x in flags if x.get('plane_mixed')),
                'n_imdd_osnr_conflict': sum(1 for x in flags if x.get('imdd_osnr_conflict')),
                'n_snr_ambiguous': sum(1 for x in flags if x.get('snr_ambiguous')),
                'n_dz_drmin_alias': sum(1 for x in flags if x.get('dz_drmin_alias')),
                'n_governance_blocked': sum(1 for x in flags if x.get('governance_blocked')),
            }),
            'tradeoff_points': points_by_paper.get(pid, {'n_points':0,'n_crq_eligible_points':0}),
            'anchor_count': len(anchors_by_paper.get(pid,[])),
            'anchors': anchors_by_paper.get(pid,[]),
        }
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
        conf = 'high' if rec['anchor_count']>=8 else ('medium' if rec['anchor_count']>=3 else 'low')
        cluster_rows.append({
            'paper_id': pid,
            'medium': medium,
            'task_type': task,
            'n_scenarios': rec['governance']['n_scenarios'],
            'n_crq_candidate': rec['governance']['n_crq_candidate'],
            'n_crq_eligible': rec['governance']['n_crq_eligible'],
            'n_plane_mixed': rec['governance']['n_plane_mixed'],
            'n_imdd_osnr_conflict': rec['governance']['n_imdd_osnr_conflict'],
            'n_snr_ambiguous': rec['governance']['n_snr_ambiguous'],
            'n_dz_drmin_alias': rec['governance']['n_dz_drmin_alias'],
            'n_governance_blocked': rec['governance']['n_governance_blocked'],
            'anchor_count': rec['anchor_count'],
            'confidence': conf,
        })
print('Saved:', graph)
pd.DataFrame(cluster_rows).to_csv(OUTPUT_DIR / 'cluster_map.csv', index=False)
print('Saved:', OUTPUT_DIR / 'cluster_map.csv')

# axis + mapping docs
axis_md = '\n'.join([
    '# Section 5 Axis Definitions (v3)',
    '',
    'Axis-1 Medium: normalized labels aligned with Section IV taxonomy.',
    'Axis-2 Communication utility: R (bits/s).',
    'Axis-3 Sensing utility: drmin/dz/sigma_r with CRB as auxiliary context.',
    'Axis-4 Coupling and governance: tradeoff_type/coupling_mode plus plane and aliasing controls.',
])
(OUTPUT_DIR / 'axis_definitions.md').write_text(axis_md, encoding='utf-8')
print('Saved:', OUTPUT_DIR / 'axis_definitions.md')

mapping_md = '\n'.join([
    '# Section 5 Mapping Rules (v3)',
    '',
    '1. CRQ candidate requires R and drmin eligibility.',
    '2. CRQ valid point requires: not plane_mixed, not imdd_osnr_conflict, not snr_ambiguous, not dz_drmin_alias.',
    '3. In cabled_fibre analyses, dz is valid as spatial granularity but cannot substitute drmin for CRQ.',
    '4. Label normalization follows the same medium map used in Section IV.',
    '5. Claim gate: >=1 DIRECT or >=2 INDIRECT.',
])
(OUTPUT_DIR / 'mapping_rules.md').write_text(mapping_md, encoding='utf-8')
print('Saved:', OUTPUT_DIR / 'mapping_rules.md')

# contract violations
viol = []
if not df5b.empty:
    for _,r in df5b.iterrows():
        pid = str(r.get('paper_id',''))
        if int(r.get('n_plane_mixed',0) or 0) > 0:
            viol.append({'paper_id': pid, 'section':'5B', 'category':'METRIC_PLANE', 'severity':'MAJOR', 'reason':'OSNR and SNR mixed', 'evidence': f"n_plane_mixed={int(r.get('n_plane_mixed',0) or 0)}"})
        if int(r.get('n_imdd_osnr_conflict',0) or 0) > 0:
            viol.append({'paper_id': pid, 'section':'5B', 'category':'METRIC_PLANE', 'severity':'MAJOR', 'reason':'IM/DD with OSNR conflict', 'evidence': f"n_imdd_osnr_conflict={int(r.get('n_imdd_osnr_conflict',0) or 0)}"})
        if int(r.get('n_snr_ambiguous',0) or 0) > 0:
            viol.append({'paper_id': pid, 'section':'5B', 'category':'METRIC_PLANE', 'severity':'MAJOR', 'reason':'SNR used without explicit plane cue', 'evidence': f"n_snr_ambiguous={int(r.get('n_snr_ambiguous',0) or 0)}"})
        if int(r.get('n_dz_drmin_alias',0) or 0) > 0:
            viol.append({'paper_id': pid, 'section':'5B', 'category':'METRIC_ALIASING', 'severity':'MAJOR', 'reason':'dz and drmin used as interchangeable values', 'evidence': f"n_dz_drmin_alias={int(r.get('n_dz_drmin_alias',0) or 0)}"})
if not df5p.empty:
    for _,r in df5p.iterrows():
        if to_bool(r.get('dz_drmin_alias',False)):
            viol.append({'paper_id': str(r.get('paper_id','')), 'section':'5C', 'category':'METRIC_ALIASING', 'severity':'MAJOR', 'reason':'CRQ point blocked by dz/drmin alias risk', 'evidence': f"scenario={r.get('scenario_index')}; drmin_m={r.get('drmin_m')}; dz_m={r.get('dz_m')}"})
viol_df = pd.DataFrame(viol, columns=['paper_id','section','category','severity','reason','evidence'])
viol_df.to_csv(OUTPUT_DIR / 'contract_violations.csv', index=False)
print('Saved:', OUTPUT_DIR / 'contract_violations.csv')
print('Violations:', len(viol_df))


Saved: analysis/V_ev_v2/retrieval_hits.jsonl
Saved: analysis/V_ev_v2/anchor_table.csv
Saved: analysis/V_ev_v2/evidence_graph.jsonl
Saved: analysis/V_ev_v2/cluster_map.csv
Saved: analysis/V_ev_v2/axis_definitions.md
Saved: analysis/V_ev_v2/mapping_rules.md
Saved: analysis/V_ev_v2/contract_violations.csv
Violations: 299


In [15]:
# @title 15. Readiness Report
report_files = [
    'section5A_evidence.csv',
    's5b_met_gov.csv',
    'section5C_tradeoff_points.csv',
    's5c_trade_mnts.csv',
    'section5D_modality_slices.csv',
    'section5E_pareto_points.csv',
    'section5E_summary_table.csv',
    'section5E_summary.json',
    'evidence_graph.jsonl',
    'retrieval_hits.jsonl',
    'anchor_table.csv',
    'axis_definitions.md',
    'mapping_rules.md',
    'cluster_map.csv',
    'contract_violations.csv',
]
report = []
for fname in report_files:
    p = OUTPUT_DIR / fname
    report.append(f'{fname}: ' + ('OK' if p.exists() else 'MISSING'))
stats = []
try:
    p5a = OUTPUT_DIR / 'section5A_evidence.csv'
    p5c = OUTPUT_DIR / 'section5C_tradeoff_points.csv'
    pa = OUTPUT_DIR / 'anchor_table.csv'
    if p5a.exists():
        d = pd.read_csv(p5a)
        stats.append(f'section5A_rows: {len(d)}')
        stats.append(f'section5A_unique_papers: {d["paper_id"].nunique() if "paper_id" in d.columns else 0}')
    if p5c.exists():
        d = pd.read_csv(p5c)
        stats.append(f'section5C_points_rows: {len(d)}')
    if pa.exists():
        d = pd.read_csv(pa)
        stats.append(f'anchor_rows: {len(d)}')
except Exception as e:
    stats.append(f'stats_error: {e}')
report_path = OUTPUT_DIR / 'readiness_report.md'
report_path.write_text('\n'.join(report + [''] + stats), encoding='utf-8')
print('\n'.join(report + [''] + stats))
print('Saved:', report_path)


section5A_evidence.csv: OK
s5b_met_gov.csv: OK
section5C_tradeoff_points.csv: OK
s5c_trade_mnts.csv: OK
section5D_modality_slices.csv: OK
section5E_pareto_points.csv: OK
section5E_summary_table.csv: OK
section5E_summary.json: OK
evidence_graph.jsonl: OK
retrieval_hits.jsonl: OK
anchor_table.csv: OK
axis_definitions.md: OK
mapping_rules.md: OK
cluster_map.csv: OK
contract_violations.csv: OK

section5A_rows: 2352
section5A_unique_papers: 221
section5C_points_rows: 226
anchor_rows: 2540
Saved: analysis/V_ev_v2/readiness_report.md
